# 01 — Data Exploration

**ICS555 Capstone: African Landmark Recognition**

This notebook explores the landmark dataset:
- Class distribution and imbalance analysis
- Sample images per class
- Dataset statistics (mean / std)
- Augmentation visualisation (full vs minimal)

In [ ]:
# ── Colab setup ────────────────────────────────────────────────────────────
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/ajegetina/african-landmark-recognition.git
    %cd african-landmark-recognition
    !pip install -q -r requirements.txt

    # Mount Drive if dataset lives there
    from google.colab import drive
    drive.mount('/content/drive')
    # Symlink the dataset so helpers.py can find it
    !ln -sf /content/drive/MyDrive/landmark_images landmark_images

In [ ]:
%matplotlib inline
import sys, math
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import datasets, transforms

sys.path.insert(0, '..')
from src.helpers import compute_mean_and_std, get_data_location

DATA_PATH = Path(get_data_location())

## 1. Class distribution

In [ ]:
train_ds = datasets.ImageFolder(DATA_PATH / 'train')
test_ds  = datasets.ImageFolder(DATA_PATH / 'test')

class_names = train_ds.classes
n_classes = len(class_names)
print(f'Classes: {n_classes}')
print(f'Train images: {len(train_ds)}')
print(f'Test  images: {len(test_ds)}')

In [ ]:
train_counts = Counter(label for _, label in train_ds.samples)
test_counts  = Counter(label for _, label in test_ds.samples)

sorted_labels = sorted(train_counts.keys(), key=lambda x: -train_counts[x])
train_vals = [train_counts[l] for l in sorted_labels]
test_vals  = [test_counts[l]  for l in sorted_labels]
names_sorted = [class_names[l] for l in sorted_labels]

fig, ax = plt.subplots(figsize=(20, 4))
x = np.arange(n_classes)
ax.bar(x - 0.2, train_vals, width=0.4, label='Train', color='steelblue')
ax.bar(x + 0.2, test_vals,  width=0.4, label='Test',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels(names_sorted, rotation=90, fontsize=7)
ax.set_ylabel('# Images')
ax.set_title('Per-class image counts (sorted by train count)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nMin train: {min(train_vals)}  Max train: {max(train_vals)}  Ratio: {max(train_vals)/min(train_vals):.1f}x')

## 2. Dataset mean and std

In [ ]:
mean, std = compute_mean_and_std()
print(f'Mean: {mean.tolist()}')
print(f'Std:  {std.tolist()}')

## 3. Sample images per class

In [ ]:
from PIL import Image

n_show = 5
# Show n_show images from 6 random classes
rng = np.random.default_rng(42)
sample_classes = rng.choice(n_classes, size=6, replace=False)

fig, axes = plt.subplots(6, n_show, figsize=(n_show * 3, 6 * 3))
for row, cls_idx in enumerate(sample_classes):
    paths = [p for p, l in train_ds.samples if l == cls_idx]
    chosen = rng.choice(paths, size=min(n_show, len(paths)), replace=False)
    for col, p in enumerate(chosen):
        img = Image.open(p).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(class_names[cls_idx], fontsize=9, loc='left')
plt.suptitle('Sample training images', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Augmentation comparison: full vs minimal

In [ ]:
from torchvision import transforms
import torch

full_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
])

minimal_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

# Pick one image
sample_path, sample_cls = train_ds.samples[0]
img = Image.open(sample_path).convert('RGB')

n_versions = 5
fig, axes = plt.subplots(2, n_versions + 1, figsize=((n_versions + 1) * 3, 6))

axes[0, 0].imshow(img)
axes[0, 0].set_title('Original', fontsize=9)
axes[1, 0].imshow(img)
axes[1, 0].set_title('Original', fontsize=9)

for i in range(n_versions):
    t_full = full_transform(img).permute(1, 2, 0).numpy().clip(0, 1)
    axes[0, i + 1].imshow(t_full)
    axes[0, i + 1].set_title(f'Full aug #{i+1}', fontsize=8)

    t_min = minimal_transform(img).permute(1, 2, 0).numpy().clip(0, 1)
    axes[1, i + 1].imshow(t_min)
    axes[1, i + 1].set_title(f'Minimal aug #{i+1}', fontsize=8)

for ax in axes.flatten(): ax.axis('off')
axes[0, 0].set_ylabel('Full augmentation', fontsize=10)
axes[1, 0].set_ylabel('Minimal augmentation', fontsize=10)
plt.suptitle(f'Class: {class_names[sample_cls]}', fontsize=12)
plt.tight_layout()
plt.show()